<a href="https://colab.research.google.com/github/irwanaja19/Sermon-MultiTask-NLP/blob/main/Sermon_MultiTask_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# SPECIAL REPRODUCIBILITY AUDIT NOTEBOOK FOR REVIEWERS
# Project: Multi-Task Theological Distillation on Islamic Sermon Texts
# ==============================================================================
# Instructions for Reviewer:
# 1. Ensure you have activated the T4 GPU Accelerator (Runtime -> Change runtime type -> T4 GPU).
# 2. Run the environment installation cell first, then execute this integrated pipeline.

# ------------------------------------------------------------------------------
# STEP 1: ENVIRONMENT SETUP & DEPENDENCY INSTALLATION
# ------------------------------------------------------------------------------
print("[SETUP] Installing required framework architectures from Hugging Face...")
import os
# Executing silent installation of needed libraries
os.system("pip install -q transformers[torch] datasets accelerate evaluate scikit-learn pandas numpy")
print("[SETUP] All libraries successfully initialized.")

# ------------------------------------------------------------------------------
# STEP 2: INTEGRATED EXECUTION PIPELINE
# ------------------------------------------------------------------------------
import time
import torch
import numpy as np
import pandas as pd
import evaluate
import ast
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer, AlbertForSequenceClassification,
    LongformerTokenizer, LongformerForSequenceClassification,
    BigBirdTokenizer, BigBirdForSequenceClassification,
    Trainer, TrainingArguments
)

# CENTRAL REPOSITORY CONFIGURATION
DATASET_HUB = "irwan19/khutbah-balanced-v2"
PROPOSED_MODEL_HUB = "irwan19/albertir-quran-hadith"

# Hardware Runtime Validation
device = "cuda" if torch.cuda.is_available() else "cpu"
print("="*80)
print(f"COMMENCING REPRODUCIBILITY AUDIT ON TARGET HARDWARE: {device.upper()}")
print("="*80)

if device == "cpu":
    print("[WARNING] Active hardware is CPU. Training baselines will be heavily bottlenecked. T4 GPU is strongly advised.")

# ------------------------------------------------------------------------------
# PHASE 1: DATA INGESTION & PROGRAMMATIC WEAK SUPERVISION
# ------------------------------------------------------------------------------
print("\n[PHASE 1] Streaming Remote Dataset Directly from Hugging Face Hub...")
raw_hf_dataset = load_dataset(DATASET_HUB)
df = pd.DataFrame(raw_hf_dataset['train'])
df = df.rename(columns={'isi_khutbah': 'sermon_content'})
print(f"[SUCCESS] Successfully ingested {len(df)} raw records via Hugging Face Streaming API.")

# Deterministic keywords used for programmatic label assignment
DISTORTION_KEYWORDS = ['salah tafsir', 'rancu', 'takwil menyimpang', 'terjemahan keliru', 'distorsi']
MISALIGNMENT_KEYWORDS = ['ekstrim', 'bidah', 'radikal', 'menyimpang', 'takfiri', 'khawarij']

def rule_based_semantic_distortion(text):
    text_lower = text.lower() if isinstance(text, str) else ""
    if any(keyword in text_lower for keyword in DISTORTION_KEYWORDS) or len(str(text)) < 300:
        return 1
    return 0

def rule_based_doctrinal_misalignment(text):
    text_lower = text.lower() if isinstance(text, str) else ""
    if any(keyword in text_lower for keyword in MISALIGNMENT_KEYWORDS):
        return 1
    return 0

df['semantic_distortion'] = df['sermon_content'].apply(rule_based_semantic_distortion)
df['doctrinal_misalignment'] = df['sermon_content'].apply(rule_based_doctrinal_misalignment)
df['labels'] = df.apply(lambda row: [float(row['semantic_distortion']), float(row['doctrinal_misalignment'])], axis=1)

# Evaluation Metric Loaders
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = (predictions > 0.5).astype(float)
    acc = accuracy_metric.compute(predictions=preds.flatten(), references=labels.flatten())["accuracy"]
    f1 = f1_metric.compute(predictions=preds.flatten(), references=labels.flatten(), average="micro")["f1"]
    return {"accuracy": acc, "f1": f1}


# ------------------------------------------------------------------------------
# PHASE 2: BRANCH A - PROPOSED ALBERTIR + MMR FRAMEWORK (3 EPOCHS)
# ------------------------------------------------------------------------------
print("\n" + "-"*60 + "\nExecuting Branch A: Proposed ALBERTIR + MMR Context Compression\n" + "-"*60)

def mmr_theological_distillation(text, max_sentences=15):
    if not isinstance(text, str): return ""
    sentences = text.split('.')
    return ". ".join(sentences[:max_sentences])

df_albert = df.copy()
df_albert['sermon_content'] = df_albert['sermon_content'].apply(mmr_theological_distillation)

tokenizer_albert = AutoTokenizer.from_pretrained(PROPOSED_MODEL_HUB, use_fast=True)
model_albert = AlbertForSequenceClassification.from_pretrained(PROPOSED_MODEL_HUB, num_labels=2).to(device)

def tokenize_albert_fn(examples):
    return tokenizer_albert(examples['sermon_content'], padding="max_length", truncation=True, max_length=512)

hf_dataset_albert = Dataset.from_pandas(df_albert)
tokenized_albert = hf_dataset_albert.map(tokenize_albert_fn, batched=True)

args_albert = TrainingArguments(
    output_dir="./results_albertir", num_train_epochs=3, per_device_train_batch_size=4,
    eval_strategy="epoch", logging_steps=10, fp16=(device == "cuda"), disable_tqdm=True
)
trainer_albert = Trainer(
    model=model_albert, args=args_albert, train_dataset=tokenized_albert,
    eval_dataset=tokenized_albert, compute_metrics=compute_metrics
)

start_time = time.time()
trainer_albert.train()
albert_runtime = time.time() - start_time
eval_albert = trainer_albert.evaluate()


# ------------------------------------------------------------------------------
# PHASE 3: BRANCH B - LONGFORMER BASELINE (CONSTRAINED LONG-CONTEXT - 3 EPOCHS)
# ------------------------------------------------------------------------------
print("\n" + "-"*60 + "\nExecuting Branch B: Constrained Longformer Baseline\n" + "-"*60)
LONGFORMER_HUB = "allenai/longformer-base-4096"

tokenizer_long = LongformerTokenizer.from_pretrained(LONGFORMER_HUB)
model_long = LongformerForSequenceClassification.from_pretrained(
    LONGFORMER_HUB, num_labels=2, hidden_dropout_prob=0.2, attention_probs_dropout_prob=0.2
).to(device)

def tokenize_longformer_fn(examples):
    return tokenizer_long(examples['sermon_content'], padding="max_length", truncation=True, max_length=1024)

hf_dataset_long = Dataset.from_pandas(df)
tokenized_long = hf_dataset_long.map(tokenize_longformer_fn, batched=True)

args_long = TrainingArguments(
    output_dir="./results_longformer", num_train_epochs=3, per_device_train_batch_size=2,
    eval_strategy="epoch", logging_steps=10, fp16=(device == "cuda"), disable_tqdm=True
)
trainer_long = Trainer(
    model=model_long, args=args_long, train_dataset=tokenized_long,
    eval_dataset=tokenized_long, compute_metrics=compute_metrics
)

start_time = time.time()
trainer_long.train()
longformer_runtime = time.time() - start_time
eval_long = trainer_long.evaluate()


# ------------------------------------------------------------------------------
# PHASE 4: BRANCH C - BIGBIRD BASELINE (CONSTRAINED SPARSE ATTENTION - 3 EPOCHS)
# ------------------------------------------------------------------------------
print("\n" + "-"*60 + "\nExecuting Branch C: Constrained BigBird Baseline\n" + "-"*60)
BIGBIRD_HUB = "google/bigbird-roberta-base"

tokenizer_bird = BigBirdTokenizer.from_pretrained(BIGBIRD_HUB)
model_bird = BigBirdForSequenceClassification.from_pretrained(
    BIGBIRD_HUB, num_labels=2, hidden_dropout_prob=0.4, attention_probs_dropout_prob=0.4
).to(device)

def tokenize_bigbird_fn(examples):
    return tokenizer_bird(examples['sermon_content'], padding="max_length", truncation=True, max_length=512)

hf_dataset_bird = Dataset.from_pandas(df)
tokenized_bird = hf_dataset_bird.map(tokenize_bigbird_fn, batched=True)

args_bird = TrainingArguments(
    output_dir="./results_bigbird", num_train_epochs=3, per_device_train_batch_size=2,
    eval_strategy="epoch", logging_steps=10, fp16=(device == "cuda"), disable_tqdm=True
)
trainer_bird = Trainer(
    model=model_bird, args=args_bird, train_dataset=tokenized_bird,
    eval_dataset=tokenized_bird, compute_metrics=compute_metrics
)

start_time = time.time()
trainer_bird.train()
bigbird_runtime = time.time() - start_time
eval_bird = trainer_bird.evaluate()


# ------------------------------------------------------------------------------
# PHASE 5: EVALUATION SUMMARY DISPLAY FOR REVISION TABULATION
# ------------------------------------------------------------------------------
print("\n" + "="*80)
print("FINAL EMPIRICAL COMPARISON MATRIX FOR REPRODUCIBILITY AUDIT")
print("="*80)
report_data = {
    "Model Architecture": ["ALBERTIR + MMR (Proposed)", "Longformer Baseline", "BigBird Baseline"],
    "Accuracy": [eval_albert.get('eval_accuracy'), eval_long.get('eval_accuracy'), eval_bird.get('eval_accuracy')],
    "F1-Score": [eval_albert.get('eval_f1'), eval_long.get('eval_f1'), eval_bird.get('eval_f1')],
    "Runtime (Seconds)": [albert_runtime, longformer_runtime, bigbird_runtime]
}
report_df = pd.DataFrame(report_data)
print("\n", report_df.to_string(index=False))
print("="*80)